# FDR GPT-2 Fine-Tuning

Fine-tunes GPT-2 on the UCSB American Presidency Project FDR corpus to generate:
1. **Policy-grounded responses** — FDR-style text steered by policy domain (economy, labor, foreign policy, etc.)
2. **Iconic quotes** — short, punchy, quotable FDR-style lines

**Before running:**
1. Runtime > Change runtime type > **T4 GPU**
2. Upload your `fdr_ucsb/txt/` folder to Google Drive

**Sections:**
1. Setup & install
2. Load & inspect UCSB corpus
3. Policy domain classification
4. Extract iconic passages
5. Train / eval split
6. Tokenize
7. Fine-tune GPT-2
8. Generate: policy-grounded responses
9. Generate: iconic quotes
10. Batch generation for eval

## 1. Setup

In [14]:
!pip install transformers datasets accelerate -q

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import os, re, random, math, glob, inspect
import numpy as np
import torch
import transformers
from collections import Counter
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer, TrainingArguments, pipeline
)

print('Transformers version:', transformers.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

Transformers version: 5.0.0
GPU available: True
Device: Tesla T4


## 2. Load & Inspect UCSB Corpus

Reads individual `.txt` files from your Google Drive folder. Each file has a header block (Title/Date/Category/URL) followed by a `========` separator, then the speech body.

In [17]:
# ── EDIT THIS PATH to match your Google Drive folder ──
CORPUS_DIR = '/content/drive/MyDrive/fdr_ucsb/txt'
SMOKE_TEST = True
SMOKE_FILE_LIMIT = 100

txt_files = sorted(glob.glob(os.path.join(CORPUS_DIR, '*.txt')))
selected_txt_files = txt_files[:SMOKE_FILE_LIMIT] if SMOKE_TEST else txt_files
print(f'Found {len(txt_files)} text files in {CORPUS_DIR}')
if SMOKE_TEST:
    print(f'Smoke test: using {len(selected_txt_files)} of {len(txt_files)} discovered files')
else:
    print(f'Full run: using all {len(selected_txt_files)} discovered files')
if selected_txt_files:
    print(f'First selected: {os.path.basename(selected_txt_files[0])}')
    print(f'Last selected:  {os.path.basename(selected_txt_files[-1])}')

Found 2853 text files in /content/drive/MyDrive/fdr_ucsb/txt
Smoke test: using 100 of 2853 discovered files
First selected: 0001_the_governor_enters_the_first_primary_campaign_for_the_presidential_nomination.txt
Last selected:  0100_executive_order_6105appointment_of_miss_jessie_b_saunders.txt


In [18]:
SEPARATOR = '=' * 72

def parse_ucsb_file(filepath):
    """Parse a single UCSB speech file into metadata + body text."""
    with open(filepath, 'r', encoding='utf-8') as f:
        raw = f.read()

    # Split on the ======== separator
    parts = raw.split(SEPARATOR, 1)
    header = parts[0] if len(parts) > 1 else ''
    body = parts[1].strip() if len(parts) > 1 else raw.strip()

    # Parse header fields (UCSB format: "Title:    value")
    meta = {}
    for line in header.split('\n'):
        line = line.strip()
        if line.startswith('Title:'):
            meta['title'] = line.split(':', 1)[1].strip()
        elif line.startswith('Date:'):
            meta['date'] = line.split(':', 1)[1].strip()
        elif line.startswith('Category:'):
            meta['category'] = line.split(':', 1)[1].strip()
        elif line.startswith('URL:'):
            meta['url'] = line.split(':', 1)[1].strip()

    return {'meta': meta, 'text': body, 'file': os.path.basename(filepath)}

# Parse only the selected files for this run
documents = []
skipped = 0
print(f'About to parse {len(selected_txt_files)} files')
for fp in selected_txt_files:
    doc = parse_ucsb_file(fp)
    # Skip very short documents (< 50 words)
    if len(doc['text'].split()) >= 50:
        documents.append(doc)
    else:
        skipped += 1

print(f'Parsed {len(documents)} documents from {len(selected_txt_files)} selected files ({skipped} skipped as too short)')
print(f'Sample title: {documents[0]["meta"].get("title", "?")}')
print(f'Sample date:  {documents[0]["meta"].get("date", "?")}')

About to parse 100 files
Parsed 94 documents from 100 selected files (6 skipped as too short)
Sample title: The Governor Enters the First Primary Campaign for the Presidential Nomination
Sample date:  January 22, 1932


In [19]:
word_counts = [len(d['text'].split()) for d in documents]
print(f'Documents        : {len(documents)}')
print(f'Total words      : {sum(word_counts):,}')
print(f'Avg words/doc    : {int(np.mean(word_counts)):,}')
print(f'Median words/doc : {int(np.median(word_counts)):,}')
print(f'Min words/doc    : {min(word_counts)}')
print(f'Max words/doc    : {max(word_counts):,}')

cats = Counter(d['meta'].get('category', 'Unknown') for d in documents)
print('\nCategories:')
for cat, count in cats.most_common(20):
    print(f'  {count:4d}  {cat}')

# Date range
dates = [d['meta'].get('date', '') for d in documents if d['meta'].get('date')]
if dates:
    print(f'\nDate range: {dates[0]} — {dates[-1]}')

Documents        : 94
Total words      : 139,281
Avg words/doc    : 1,481
Median words/doc : 554
Min words/doc    : 65
Max words/doc    : 8,929

Categories:
    30  Executive Orders
    24  Campaign Documents
    10  Messages
     7  Proclamations
     6  Transition Documents
     3  Letters
     3  News Conferences
     3  Statements by the Press Secretary
     2  Spoken Addresses and Remarks
     2  Statements
     1  Inaugural Addresses
     1  Timeline
     1  Press Releases
     1  Fireside Chats

Date range: January 22, 1932 — April 06, 1933


## 3. Policy Domain Classification

Tag each speech with a policy domain so the model learns to associate FDR's voice with specific topics. At generation time, you steer output by prompting with `[POLICY: Economy]` etc.

Domains are based on FDR's actual presidency:
- **Economy** — banking, depression, recovery, spending
- **Labor** — workers, wages, unions, employment
- **Foreign Policy** — war, allies, neutrality, defense
- **Social Welfare** — relief, Social Security, poverty, housing
- **Agriculture** — farms, crops, rural America
- **Government** — democracy, Constitution, reform, courts
- **Conservation** — land, resources, environment, CCC

In [20]:
# ── Policy domain keyword classifier ──
POLICY_DOMAINS = {
    'Economy': [
        r'\bbank', r'\bdepress', r'\brecovery\b', r'\bbudget\b', r'\btax',
        r'\binflation\b', r'\bcredit\b', r'\bcurrency\b', r'\bwall street\b',
        r'\bspending\b', r'\bprosperity\b', r'\brecession\b', r'\bnra\b',
        r'\beconom', r'\bbusiness\b', r'\bindustr', r'\btariff',
    ],
    'Labor': [
        r'\bworker', r'\bwage', r'\bunion', r'\blabor\b', r'\bemploy',
        r'\bunemploy', r'\bjob[s ]', r'\bfair labor\b', r'\bworkingm',
        r'\bsweatshop', r'\bchild labor\b', r'\bcollective bargain',
    ],
    'Foreign Policy': [
        r'\bwar\b', r'\bpeace\b', r'\ball[yi]', r'\bnazi', r'\bhitler',
        r'\bjapan', r'\bgerman', r'\bneutral', r'\bdefense\b', r'\barmy\b',
        r'\bnavy\b', r'\bmilitar', r'\bweapon', r'\barsenal\b', r'\btyrann',
        r'\bdemocrac.*abroad', r'\bfascis', r'\baggress', r'\binvasion',
        r'\bunited nations\b', r'\blend.lease', r'\batlantic charter',
    ],
    'Social Welfare': [
        r'\brelief\b', r'\bsocial security', r'\bpoverty\b', r'\bhousing\b',
        r'\bwelfare\b', r'\bpension', r'\binsurance\b', r'\bhungr',
        r'\bhomeless', r'\bdestitu', r'\borphan', r'\bwpa\b', r'\bccc\b',
    ],
    'Agriculture': [
        r'\bfarm', r'\bcrop', r'\bagricultur', r'\brural\b', r'\bsoil\b',
        r'\bdrought\b', r'\bharvest', r'\bplant', r'\blivestock',
        r'\birrigat', r'\bdust bowl',
    ],
    'Government': [
        r'\bdemocracy\b', r'\bconstitution', r'\breform\b', r'\bcourt',
        r'\bcongress\b', r'\bsupreme court', r'\bvot', r'\belect',
        r'\brepublic\b', r'\bliberty\b', r'\bfreedom\b', r'\bright[s ]\b',
    ],
    'Conservation': [
        r'\bconserv', r'\bland\b', r'\bforest', r'\briver', r'\bwater\b',
        r'\bresource', r'\benviron', r'\bnational park', r'\bwildlife',
        r'\btva\b', r'\bdam\b', r'\berosion\b',
    ],
}

def classify_policy(text, top_n=2):
    """Return top N policy domains by keyword hit count."""
    text_lower = text.lower()
    scores = {}
    for domain, patterns in POLICY_DOMAINS.items():
        score = sum(len(re.findall(p, text_lower)) for p in patterns)
        if score > 0:
            scores[domain] = score
    if not scores:
        return ['General']
    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked[:top_n]

# Classify every document
for doc in documents:
    doc['domains'] = classify_policy(doc['text'])
    doc['primary_domain'] = doc['domains'][0]

# Show distribution
domain_counts = Counter(d['primary_domain'] for d in documents)
print('Policy domain distribution (primary):')
for domain, count in domain_counts.most_common():
    bar = '#' * (count // 2)
    print(f'  {domain:<20} {count:4d}  {bar}')

print(f'\nTotal classified: {len(documents)}')

Policy domain distribution (primary):
  Economy                40  ####################
  Government             15  #######
  Foreign Policy         14  #######
  Social Welfare          7  ###
  Agriculture             6  ###
  General                 5  ##
  Labor                   4  ##
  Conservation            3  #

Total classified: 94


## 4. Extract Iconic Passages

Pull out short, punchy paragraphs that have high rhetorical density — tricolons, moral framing, direct address. These get tagged `[ICONIC]` in training data so the model learns what "quotable FDR" sounds like, separate from long policy speeches.

In [21]:
# ── Rhetorical pattern detectors ──
RHETORICAL_PATTERNS = [
    r'\bwe must\b',
    r'\blet us\b',
    r'\bI tell you\b',
    r'\bmy friends\b',
    r'\bthis generation\b',
    r'\bthis nation\b',
    r'\bwe have nothing to fear\b',
    r'\bthe only thing\b',
    r'\bI pledge\b',
    r'\bI ask\b',
    r'\bduty\b',
    r'\bdestiny\b',
    r'\bfreedom\b',
    r'\bliberty\b',
    r'\bjustice\b',
    r'\bfaith\b',
    r'\bcourage\b',
    r'\bsacrifice\b',
    r'\bnot .* but\b',           # "not X but Y" contrast
    r'\b\w+, \w+, and \w+\b',   # tricolon pattern
]

def rhetorical_score(paragraph):
    """Score a paragraph's rhetorical density (0-1)."""
    text_lower = paragraph.lower()
    hits = sum(1 for p in RHETORICAL_PATTERNS if re.search(p, text_lower))
    word_count = len(paragraph.split())
    if word_count == 0:
        return 0
    # Normalize: hits per 50 words, capped at 1.0
    return min(hits / max(word_count / 50, 1), 1.0)

def extract_iconic_passages(documents, min_words=15, max_words=120, min_score=0.3):
    """Extract short, rhetorically dense paragraphs from all documents."""
    iconic = []
    for doc in documents:
        paragraphs = doc['text'].split('\n\n')
        for para in paragraphs:
            para = para.strip()
            wc = len(para.split())
            if wc < min_words or wc > max_words:
                continue
            score = rhetorical_score(para)
            if score >= min_score:
                iconic.append({
                    'text': para,
                    'score': score,
                    'word_count': wc,
                    'source_title': doc['meta'].get('title', '?'),
                    'domain': doc['primary_domain'],
                })
    return sorted(iconic, key=lambda x: x['score'], reverse=True)

iconic_passages = extract_iconic_passages(documents)
print(f'Extracted {len(iconic_passages)} iconic passages')
print(f'\nTop 10 by rhetorical score:')
for i, p in enumerate(iconic_passages[:10]):
    preview = p['text'][:100].replace('\n', ' ')
    print(f'  {i+1}. [{p["score"]:.2f}] [{p["domain"]}] {preview}...')

Extracted 353 iconic passages

Top 10 by rhetorical score:
  1. [1.00] [Government] It is the simple duty of any American to serve in public position if called upon. One who believes i...
  2. [1.00] [Government] Were I now to divert my efforts in any degree by personal efforts in furtherance of my own political...
  3. [1.00] [Government] I know that you will understand the good faith in which I tell you this; and also my hope that our p...
  4. [1.00] [Government] I am grateful that my friends of North Dakota wish to present my name in the primary elections for t...
  5. [1.00] [Economy] It is high time to get back to fundamentals. It is high time to admit with courage that we are in th...
  6. [1.00] [Economy] Andrew Jackson had it. I like his blunt statement that "the spirit of equity requires that the great...
  7. [1.00] [Economy] I say with Lincoln, "Having thus chosen our course, without guile and with pure purpose, let us rene...
  8. [1.00] [Economy] We need enthusiasm, imagi

## 5. Train / Eval Split

Shuffle by whole document, then format training data with:
- `[POLICY: Domain]` prefix on full speeches → teaches policy-grounded generation
- `[ICONIC]` prefix on extracted passages → teaches quotable, punchy output

Both tag types become sterable prompts at generation time.

In [22]:
SEED       = 42
EVAL_RATIO = 0.10  # 90/10 split
EOS        = '<|endoftext|>'

random.seed(SEED)
shuffled = documents.copy()
random.shuffle(shuffled)

split_idx  = int(len(shuffled) * (1 - EVAL_RATIO))
train_docs = shuffled[:split_idx]
eval_docs  = shuffled[split_idx:]

# Also split iconic passages (keep them aligned with their source docs)
train_titles = {d['meta'].get('title') for d in train_docs}
train_iconic = [p for p in iconic_passages if p['source_title'] in train_titles]
eval_iconic  = [p for p in iconic_passages if p['source_title'] not in train_titles]

print(f'Train documents : {len(train_docs)}')
print(f'Eval documents  : {len(eval_docs)}')
print(f'Train words     : {sum(len(d["text"].split()) for d in train_docs):,}')
print(f'Eval words      : {sum(len(d["text"].split()) for d in eval_docs):,}')
print(f'Train iconic    : {len(train_iconic)} passages')
print(f'Eval iconic     : {len(eval_iconic)} passages')

Train documents : 84
Eval documents  : 10
Train words     : 103,223
Eval words      : 36,058
Train iconic    : 267 passages
Eval iconic     : 86 passages


### Smoke Test Mode

Smoke-test selection now happens at file discovery time, so parsing, classification, extraction, and training all use the same subset. Change `SMOKE_TEST` or `SMOKE_FILE_LIMIT` in the corpus-loading cell above.

In [23]:
if SMOKE_TEST:
    print('SMOKE TEST MODE ENABLED')
    print(f'  Selected files : {len(selected_txt_files)}')
    print(f'  File limit     : {SMOKE_FILE_LIMIT}')
else:
    print('Full dataset mode enabled')

SMOKE TEST MODE ENABLED
  Selected files : 100
  File limit     : 100


In [24]:
def build_training_rows(docs, iconic):
    """Build one row per training unit so preprocessing stays inspectable."""
    rows = []

    # Full speeches with policy domain tags
    for doc in docs:
        rows.append({
            'text': f'[POLICY: {doc["primary_domain"]}] {doc["text"]}',
            'kind': 'policy',
            'domain': doc['primary_domain'],
            'title': doc['meta'].get('title', ''),
        })

    # Iconic passages with [ICONIC] tag to teach punchy quotable style
    for p in iconic:
        rows.append({
            'text': f'[ICONIC] {p["text"]}',
            'kind': 'iconic',
            'domain': 'ICONIC',
            'title': p['source_title'],
        })

    random.shuffle(rows)
    return rows

train_rows = build_training_rows(train_docs, train_iconic)
eval_rows  = build_training_rows(eval_docs, eval_iconic)

raw_datasets = DatasetDict({
    'train': Dataset.from_list(train_rows),
    'eval': Dataset.from_list(eval_rows),
})

print(f'Train rows: {len(raw_datasets["train"]):,}')
print(f'Eval rows : {len(raw_datasets["eval"]):,}')
if SMOKE_TEST:
    print('Dataset size reflects smoke-test limits above.')
print('\nFirst training row:')
print(raw_datasets['train'][0]['text'][:500])

Train rows: 351
Eval rows : 96
Dataset size reflects smoke-test limits above.

First training row:
[POLICY: Foreign Policy] CANAL ZONE


By virtue of the authority vested in me by law, it is hereby ordered that Buildings Nos. 1113, 1409, and 1411, located at Cristobal, Canal Zone, that were transferred, exclusive of the sites thereof but including free use of such sites, by Executive Order No. 3917, dated October 16, 1923, from the Panama Canal to the War Department, be, and the same are hereby, transferred under like conditions to the Navy Department.


FRANKLIN D. ROOSEVELT


The White Hous


## 6. Tokenize

GPT-2's tokenizer handles our `[POLICY: ...]` and `[ICONIC]` tags as regular tokens. We tokenize explicit dataset rows, then concatenate and chunk them into fixed-length training blocks so the preprocessing stays visible and easy to debug.

In [25]:
MODEL_NAME = 'openai-community/gpt2'   # swap for 'openai-community/gpt2-medium' if you want more capacity
BLOCK_SIZE = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples['text'])

tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets['train'].column_names,
)

def group_texts(examples):
    # Concatenate tokenized rows, then slice into uniform LM training blocks.
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated['input_ids'])
    total_length = (total_length // BLOCK_SIZE) * BLOCK_SIZE

    result = {
        k: [t[i:i + BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
        for k, t in concatenated.items()
    }
    result['labels'] = result['input_ids'].copy()
    return result

lm_datasets = tokenized_datasets.map(group_texts, batched=True)
train_dataset = lm_datasets['train']
eval_dataset  = lm_datasets['eval']

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(f'Train blocks : {len(train_dataset)}')
print(f'Eval blocks  : {len(eval_dataset)}')

sample = tokenizer.decode(train_dataset[0]['input_ids'][:80])
print(f'\nSample decoded block (first 80 tokens):')
print(sample)

Map:   0%|          | 0/351 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (5716 > 1024). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/96 [00:00<?, ? examples/s]

Map:   0%|          | 0/351 [00:00<?, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

Train blocks : 288
Eval blocks  : 110

Sample decoded block (first 80 tokens):
[POLICY: Foreign Policy] CANAL ZONE


By virtue of the authority vested in me by law, it is hereby ordered that Buildings Nos. 1113, 1409, and 1411, located at Cristobal, Canal Zone, that were transferred, exclusive of the sites thereof but including free use of such sites, by Executive Order No. 3917, dated October 16,


## 7. Fine-Tune GPT-2

For the smoke test, keep this fast and cheap. For the full run on an A100, the defaults below target a reliable run with Drive checkpoints and roughly under an hour of training for `gpt2`.

In [26]:
OUTPUT_DIR = '/content/drive/MyDrive/fdr_gpt2_smoke_model' if SMOKE_TEST else '/content/drive/MyDrive/fdr_gpt2_model'
RESUME_FROM_CHECKPOINT = None  # e.g. '/content/drive/MyDrive/fdr_gpt2_model/checkpoint-400'

# A100-friendly full-run defaults. Smoke mode stays intentionally tiny.
FULL_RUN_EPOCHS = 2
FULL_RUN_BATCH_SIZE = 8
FULL_RUN_GRAD_ACCUM = 2
FULL_RUN_SAVE_STEPS = 200
FULL_RUN_EVAL_STEPS = 200
FULL_RUN_LOG_STEPS = 25

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
use_bf16 = torch.cuda.is_available() and ('A100' in device_name)
use_fp16 = torch.cuda.is_available() and not use_bf16

training_kwargs = {
    'output_dir': OUTPUT_DIR,
    'num_train_epochs': 1 if SMOKE_TEST else FULL_RUN_EPOCHS,
    'per_device_train_batch_size': 4 if SMOKE_TEST else FULL_RUN_BATCH_SIZE,
    'per_device_eval_batch_size': 4 if SMOKE_TEST else FULL_RUN_BATCH_SIZE,
    'gradient_accumulation_steps': 2 if SMOKE_TEST else FULL_RUN_GRAD_ACCUM,
    'logging_steps': 10 if SMOKE_TEST else FULL_RUN_LOG_STEPS,
    'report_to': 'none',
    'learning_rate': 5e-5,
    'warmup_steps': 20 if SMOKE_TEST else 100,
    'weight_decay': 0.01,
    'prediction_loss_only': True,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_loss',
    'save_total_limit': 2,
}

supported_args = set(inspect.signature(TrainingArguments.__init__).parameters)

if 'fp16' in supported_args:
    training_kwargs['fp16'] = use_fp16
if 'bf16' in supported_args:
    training_kwargs['bf16'] = use_bf16
if 'overwrite_output_dir' in supported_args:
    training_kwargs['overwrite_output_dir'] = True
if 'save_strategy' in supported_args:
    training_kwargs['save_strategy'] = 'epoch' if SMOKE_TEST else 'steps'
if 'save_steps' in supported_args and not SMOKE_TEST:
    training_kwargs['save_steps'] = FULL_RUN_SAVE_STEPS
if 'evaluation_strategy' in supported_args:
    training_kwargs['evaluation_strategy'] = 'epoch' if SMOKE_TEST else 'steps'
elif 'eval_strategy' in supported_args:
    training_kwargs['eval_strategy'] = 'epoch' if SMOKE_TEST else 'steps'
if 'eval_steps' in supported_args and not SMOKE_TEST:
    training_kwargs['eval_steps'] = FULL_RUN_EVAL_STEPS

training_args = TrainingArguments(**training_kwargs)

print(f'Output dir: {OUTPUT_DIR}')
print(f'Device    : {device_name}')
print(f'Epochs    : {training_args.num_train_epochs}')
print(f'Batch size: {training_args.per_device_train_batch_size}')
print(f'Grad accum: {training_args.gradient_accumulation_steps}')
print(f'Log steps : {training_args.logging_steps}')
print(f'bf16      : {use_bf16}')
print(f'fp16      : {use_fp16}')
print('TrainingArguments API keys detected:', ', '.join(sorted(k for k in ['overwrite_output_dir', 'evaluation_strategy', 'eval_strategy', 'save_strategy', 'save_steps', 'eval_steps', 'bf16', 'fp16'] if k in supported_args)))

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)


Output dir: /content/drive/MyDrive/fdr_gpt2_smoke_model
Epochs    : 1
Log steps : 10
TrainingArguments API keys detected: eval_strategy, save_strategy


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.247518,3.214115


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=36, training_loss=3.328211466471354, metrics={'train_runtime': 26.4328, 'train_samples_per_second': 10.896, 'train_steps_per_second': 1.362, 'total_flos': 75252105216000.0, 'train_loss': 3.328211466471354, 'epoch': 1.0})

In [27]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/fdr_gpt2_smoke_model


In [28]:
eval_results = trainer.evaluate()
perplexity = math.exp(eval_results['eval_loss'])
print(f"Eval loss  : {eval_results['eval_loss']:.4f}")
print(f"Perplexity : {perplexity:.2f}")
print()
print('Perplexity guide:')
print('  < 30   excellent — model has learned FDR\'s patterns well')
print('  30-60  good — usable for generation')
print('  60-100 fair — try more epochs or gpt2-medium')
print('  > 100  poor — check data quality or increase corpus size')

Eval loss  : 3.2141
Perplexity : 24.88

Perplexity guide:
  < 30   excellent — model has learned FDR's patterns well
  30-60  good — usable for generation
  60-100 fair — try more epochs or gpt2-medium
  > 100  poor — check data quality or increase corpus size


## 8. Generate: Policy-Grounded Responses

Prompt with `[POLICY: Domain]` to steer FDR into a specific policy lane. The model learned these tags during training, so it associates them with the right vocabulary and framing.

In [29]:
generator = pipeline(
    'text-generation',
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=0 if torch.cuda.is_available() else -1
)

def generate_policy(domain, opening, max_new_tokens=300, num_variants=2,
                    temperature=0.85, top_p=0.92):
    """Generate a policy-grounded FDR response in a specific domain."""
    prompt = f'[POLICY: {domain}] {opening}'
    outputs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        num_return_sequences=num_variants,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2,
    )
    print(f'=== [POLICY: {domain}] ===')
    print(f'Prompt: "{opening}"\n')
    for i, out in enumerate(outputs):
        # Strip the [POLICY: ...] tag from output for readability
        text = out['generated_text']
        text = re.sub(r'^\[POLICY: [^\]]+\]\s*', '', text)
        print(f'--- Variant {i+1} ---')
        print(text)
        print()

def generate_iconic(opening='', max_new_tokens=80, num_variants=3,
                    temperature=0.95, top_p=0.9):
    """Generate short, punchy, quotable FDR-style lines."""
    prompt = f'[ICONIC] {opening}' if opening else '[ICONIC]'
    outputs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        num_return_sequences=num_variants,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.3,
    )
    print(f'=== [ICONIC] ===')
    if opening:
        print(f'Prompt: "{opening}"\n')
    for i, out in enumerate(outputs):
        text = out['generated_text']
        text = re.sub(r'^\[ICONIC\]\s*', '', text)
        # Trim to first complete sentence for punchiness
        sentences = re.split(r'(?<=[.!?])\s+', text)
        # Take first 1-3 sentences
        quote = ' '.join(sentences[:3])
        print(f'  {i+1}. "{quote}"')
    print()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [30]:
# ── Policy-grounded examples on modern topics ──

# Economy: Congestion pricing
generate_policy('Economy',
    'My fellow Americans, the great arteries of our cities have become choked '
    'with the burden of the automobile, and it falls upon this government to')

# Labor: AI replacing jobs
generate_policy('Labor',
    'The machine has always been the servant of man, not his master. '
    'When new inventions displace the worker, it is the duty of government to')

# Foreign Policy: Global tech competition
generate_policy('Foreign Policy',
    'The democracies of the world face a new contest — not of armies '
    'but of industries and ingenuity. We must not allow the forces of')

# Social Welfare: Housing crisis
generate_policy('Social Welfare',
    'No family in America should want for shelter. The crisis of housing '
    'in our great cities is not merely an economic question but a moral one, and')

# Government: Social media and democracy
generate_policy('Government',
    'The instruments of communication, once the province of the printing press '
    'and the public square, have fallen into the hands of private enterprise, and')

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'do_sample', 'max_new_tokens', 'top_p', 'pad_token_id', 'temperature', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [POLICY: Economy] ===
Prompt: "My fellow Americans, the great arteries of our cities have become choked with the burden of the automobile, and it falls upon this government to"

--- Variant 1 ---
My fellow Americans, the great arteries of our cities have become choked with the burden of the automobile, and it falls upon this government to rescue them. It has taken many millions in advance loans for automobiles that are inadequate or not suitable; but these banks were able to provide their own loan-payers a standard price which they could pay when demanded by customers at reasonable prices without having borrowed much money."
The Bankers' Committee continued (to President Woodrow Wilson) "That is what we do now — so I will tell you something very serious about your business plan as part if nothing else," she said.(APPLAUSE.)On June 28th Congress met on one side and read from an Executive Order approved February 26th entitled, In order To Save The American Trade Association From Imme

Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [POLICY: Labor] ===
Prompt: "The machine has always been the servant of man, not his master. When new inventions displace the worker, it is the duty of government to"

--- Variant 1 ---
The machine has always been the servant of man, not his master. When new inventions displace the worker, it is the duty of government to restore him back again and at all times in a position where he can perform some duties on himself." This was also reiterated by Secretary John Adams after meeting with Wilson; for they agreed that no one should interfere unless there were something more urgently needed — such as "a practical solution" toward employment problems or public health relief programs.(12)
The present administration seeks to re-educate Americans about their role in this great nation which seems so utterly bankrupt today because its people have lost everything over time. It tries nothing but make these false promises sound like good economic policy if true (see section 2.) On July 4th we as

Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [POLICY: Foreign Policy] ===
Prompt: "The democracies of the world face a new contest — not of armies but of industries and ingenuity. We must not allow the forces of"

--- Variant 1 ---
The democracies of the world face a new contest — not of armies but of industries and ingenuity. We must not allow the forces of foreign policy to control our lives in any one direction or country at all; nor should we let them interfere with us as long ago when these powers have been taken from people who had already used their own resources, provided they could do nothing about it themselves now." [LARGE OF THE CHAMPIONSHIPS ]
This is what my friends said during last week's debate on national security : "I am convinced that by an honest approach you will win over your voters more than once again. You know this very well because I was here yesterday night for many meetings across Washington between Republican leaders including Senator Lindsay Graham (R-NH), Congressman John Conyers (D -MI), Govern

Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [POLICY: Social Welfare] ===
Prompt: "No family in America should want for shelter. The crisis of housing in our great cities is not merely an economic question but a moral one, and"

--- Variant 1 ---
No family in America should want for shelter. The crisis of housing in our great cities is not merely an economic question but a moral one, and it cannot be solved without the protection provided by national security."
The Republican leadership promised to eliminate homelessness at any cost except through "a swift recovery" — which would take years or even decades if necessary—in order that all these people could find work free from poverty; they did so on December 1st when President Abraham Lincoln announced his plan during October's Democratic convention called upon Congress to pass legislation cutting off private insurance subsidies as well as increasing public benefits beyond federal funds altogether (with no end date). This act was approved six months later after FDR signed its 

## 9. Generate: Iconic Quotes

Short, punchy, quotable. The `[ICONIC]` tag tells the model to produce the kind of condensed, rhetorically powerful lines it saw in the extracted passages. Lower `max_new_tokens` + higher `temperature` = more creative, more concise.

In [31]:
# ── Iconic quote generation ──

# Open-ended: let the model riff
generate_iconic()

# Seeded with FDR-style openings
generate_iconic('The only limit to our realization of tomorrow')
generate_iconic('We have always known that')
generate_iconic('Let it be said of this generation')
generate_iconic('The test of our progress is not')
generate_iconic('No democracy can long survive')
generate_iconic('My friends, the American people')

Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [ICONIC] ===
  1. "The world is not made of the same materials as in a certain particular city or state. It may be that each nation finds its own means to get around, but it would never do so alone; at other times and places people use different products on their homesteads than have been produced from our local marketable resources by others through labor efforts which provide greater value for us hereon instead —"
  2. "The following persons and institutions are hereby designated: (i) A representative of the American Immigration Reform Commission; or, through a person holding office for one year who is not disqualified by reason only as an administrative position in this commission but whose status shall apply to his appointment without appeal therefrom with approval from such Commissioner prior thereto. I am authorized under section 1104(b)(7), title"
  3. "1. The following is a general discussion of the relation between these matters: (a) To an extent that I can understand them

Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [ICONIC] ===
Prompt: "The only limit to our realization of tomorrow"

  1. "The only limit to our realization of tomorrow is by virtue that we must recognize it and the conditions which have resulted will produce such an orderly, free movement from any prejudice against us. SECTION 1: EXECUTIONS OF THE FEDERAL GOVERNMENT IN DEPARTURE TO REFORM IT-OF STATE POLICE AND SORCES FOR DISTILLING RECORD RENEWABLE CORPS (SECRET"
  2. "The only limit to our realization of tomorrow is not that we can afford it, but rather the need for further study. I have already set forth and stated clearly what has occurred in regard particularly a problem which lies beyond all consideration."
In other words: "I wish you would know how far this approach cannot go before they attempt another great change!" And here are some answers from my friends at Boston Consulting Group's Federal Policy Institute (PDF"
  3. "The only limit to our realization of tomorrow is that the total prosperity will continue. It may 

Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

=== [ICONIC] ===
Prompt: "Let it be said of this generation"

  1. "Let it be said of this generation that they have taken a step backward to do nothing. This is the first time in my entire history, and not only so far as I know; but there will come an end when such steps are necessary for us all."


I look forward with great pleasure from their efforts today at strengthening our defense system within Africa's borders—by restoring peace through dialogue between peoples without coercion or by opening trade"
  2. "Let it be said of this generation: I know the work and hope that we shall all come together, but my duty is to you as a country. It should not go unnoticed in your heart by any who believe there are people here with no means except their own; they will live for generations after living on food banks or private farms whose conditions may require such treatment because some have become slaves instead from which others can take advantage through"
  3. "Let it be said of this gener

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [ICONIC] ===
Prompt: "The test of our progress is not"

  1. "The test of our progress is not based on numbers but rather the fact that in this world there are some who, for lack and without merit or reason could only act as if they had been able. I am satisfied now with those few people I have met through my own endeavours to understand their situation (and thus it has come upon me all over again). We must also remember how we've suffered from a very long war which"
  2. "The test of our progress is not, I think, to depend on the present state of affairs. It rests upon a determination and method that will always have an enduring effect as well."
On January 2, 1918, President Woodrow Wilson proclaimed: "The great American Empire stands in ruin; it must be put up again if we are going forward or can only survive by another century after war has ended. We cannot allow ourselves too"
  3. "The test of our progress is not as simple to determine what the final outcome may be, but rather

Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== [ICONIC] ===
Prompt: "No democracy can long survive"

  1. "No democracy can long survive the disintegration of any kind," said Mr. Wiegert on June 4, in an address to a joint session presided over by Mrs Gensler and other Cabinet secretaries who were briefed about his remarks before they signed off for their meetings at 6:30 p.. m., when he was informed that "the present State Department may well declare war with us or its President."


In return"
  2. "No democracy can long survive without the rule of one man, and if he is to serve as a member of this group there must be in him such an idealism that nothing shall interfere with its essential purpose. A few minutes after noon on Saturday afternoon my Secretary decided upon his policy for peace; at which point I was sent by General Washington:

"To propose no further action whatsoever concerning our country except through peaceful means"
  3. "No democracy can long survive in the absence of effective representation and leadership. 

## 10. Batch Generation for Eval

Generate FDR responses to 50 modern headlines. These become the test cases for the Day 4 eval framework. Each response gets both a policy-grounded version and an iconic quote version, saved to a JSON file on Drive.

In [32]:
import json

# ── 50 modern headlines mapped to FDR policy domains ──
MODERN_HEADLINES = [
    # Economy
    {"headline": "Fed Holds Interest Rates Steady Amid Inflation Fears", "domain": "Economy",
     "opening": "My friends, the management of our currency and credit is not a matter for bankers alone —"},
    {"headline": "Wall Street Hits Record High as Main Street Struggles", "domain": "Economy",
     "opening": "We cannot measure the prosperity of a nation by the ticker tape of Wall Street, for"},
    {"headline": "Crypto Exchange Collapses, Billions in Customer Funds Missing", "domain": "Economy",
     "opening": "Once again the speculators have played fast and loose with the savings of the common man, and"},
    {"headline": "Student Loan Debt Passes $1.7 Trillion", "domain": "Economy",
     "opening": "The youth of this nation, eager to learn and to serve, should not begin their lives shackled by"},
    {"headline": "Housing Prices Rise for 12th Consecutive Month", "domain": "Economy",
     "opening": "A home is not a speculation — it is a right. When the cost of shelter exceeds the reach of the working family,"},
    # Labor
    {"headline": "AI Chatbot Replaces 700 Customer Service Workers", "domain": "Labor",
     "opening": "The machine has always been the servant of man, not his master. When new inventions displace the worker,"},
    {"headline": "Amazon Warehouse Workers Vote to Unionize", "domain": "Labor",
     "opening": "The right of workers to organize is not a privilege granted by their employers — it is"},
    {"headline": "Gig Workers Denied Benefits in Court Ruling", "domain": "Labor",
     "opening": "No man or woman who gives an honest day's work should be denied the basic protections of"},
    {"headline": "Tech Layoffs Hit 100,000 This Quarter", "domain": "Labor",
     "opening": "When great industries cast aside their workers in the name of efficiency, it falls upon"},
    {"headline": "Minimum Wage Hasn't Kept Pace with Inflation Since 2009", "domain": "Labor",
     "opening": "A wage that cannot sustain a family is not a wage — it is an injustice, and"},
    # Foreign Policy
    {"headline": "China Surpasses US in AI Patent Filings", "domain": "Foreign Policy",
     "opening": "The democracies of the world face a new contest — not of armies but of industries and ingenuity, and"},
    {"headline": "NATO Allies Debate Defense Spending Amid Rising Tensions", "domain": "Foreign Policy",
     "opening": "The peace of the world cannot rest upon the goodwill of aggressors. Our allies and our own people must"},
    {"headline": "Russia Expands Military Operations in Eastern Europe", "domain": "Foreign Policy",
     "opening": "The forces of tyranny do not respect treaties or borders. We have seen this before, and"},
    {"headline": "UN Climate Summit Ends Without Binding Agreement", "domain": "Foreign Policy",
     "opening": "The nations of the world have gathered and spoken, yet have not acted. If we cannot find the courage to"},
    {"headline": "Global Chip Shortage Threatens National Security", "domain": "Foreign Policy",
     "opening": "A nation that cannot manufacture the instruments of its own defense is a nation at the mercy of others, and"},
    # Social Welfare
    {"headline": "Homelessness Rises 12% in Major US Cities", "domain": "Social Welfare",
     "opening": "In the richest nation the world has ever known, no citizen should sleep upon the streets. The measure of our civilization is"},
    {"headline": "Mental Health Crisis Among Teens Worsens", "domain": "Social Welfare",
     "opening": "The well-being of our children is the first duty of this republic. When the young suffer in silence,"},
    {"headline": "Rural Hospitals Continue to Close Across America", "domain": "Social Welfare",
     "opening": "The farmer and the small-town family deserve the same care as the city dweller. No American should die because"},
    {"headline": "Food Banks Report Record Demand This Holiday Season", "domain": "Social Welfare",
     "opening": "In a land of abundance, hunger is not a failure of supply — it is a failure of conscience, and"},
    {"headline": "Opioid Deaths Hit New Record in 2025", "domain": "Social Welfare",
     "opening": "A plague has settled upon our communities, not of foreign origin but of our own making, and"},
    # Government / Democracy
    {"headline": "Linguistics Professor Calls Colleague 'Mansplainer' in AI Translation Clash", "domain": "Government",
     "opening": "The pursuit of knowledge demands not insult but inquiry. When the scholars of a great nation descend to"},
    {"headline": "Voter Turnout Hits Historic Low in Midterm Elections", "domain": "Government",
     "opening": "Democracy does not die in dramatic fashion — it withers when the citizen turns away from the ballot box, and"},
    {"headline": "Social Media Algorithm Blamed for Political Polarization", "domain": "Government",
     "opening": "The instruments of communication, once the province of the public square, have been captured by machines that"},
    {"headline": "Supreme Court Limits Federal Agency Power in Landmark Ruling", "domain": "Government",
     "opening": "The Constitution is a living charter, not a straitjacket. When the courts constrain the ability of the people's government to"},
    {"headline": "Trust in Government Hits All-Time Low in New Poll", "domain": "Government",
     "opening": "The faith of the people in their government is not a luxury — it is the foundation, and when that faith erodes,"},
    # Agriculture / Environment
    {"headline": "Drought Devastates Midwest Corn Crop", "domain": "Agriculture",
     "opening": "The land does not lie. When the rains fail and the soil cracks, it is not merely the farmer who suffers — it is"},
    {"headline": "Family Farms Disappear as Corporate Agriculture Expands", "domain": "Agriculture",
     "opening": "The family farm is not a relic of the past — it is the backbone of this republic, and"},
    {"headline": "Wildfire Season Breaks Records for Third Straight Year", "domain": "Conservation",
     "opening": "The forests and the rivers and the air we breathe are held in trust for future generations, and"},
    {"headline": "Clean Water Crisis Affects 2 Million Americans", "domain": "Conservation",
     "opening": "Water is life itself. When the taps of an American city run with poison, it is not merely a failure of pipes — it is"},
    {"headline": "Flooding Displaces Thousands Along Mississippi River", "domain": "Conservation",
     "opening": "The great rivers of this continent have always been both our blessing and our challenge. We must"},
    # Tech & Modern Issues (mapped to closest FDR domain)
    {"headline": "Hospital Hacked, Patient Data Held for Ransom", "domain": "Government",
     "opening": "The safety of the citizen — whether in person or in the records that bear their name — is a sacred trust, and"},
    {"headline": "Electric Vehicle Sales Surge Past 50% Market Share", "domain": "Economy",
     "opening": "The American genius for industry has always risen to meet the challenge of the age. Today that challenge is"},
    {"headline": "Teachers Strike Over Pay and Class Sizes", "domain": "Labor",
     "opening": "The teacher stands at the foundation of our democracy. When we fail to honor their service with fair compensation,"},
    {"headline": "Prescription Drug Prices Rise 30% in Five Years", "domain": "Social Welfare",
     "opening": "No American should choose between their medicine and their meals. The profiteering of the pharmaceutical industry is"},
    {"headline": "Infrastructure Bill Stalls in Congress", "domain": "Economy",
     "opening": "The bridges and roads and rails of this nation are the arteries of commerce and connection, and when they crumble,"},
    {"headline": "Immigration Debate Paralyzes Congress", "domain": "Government",
     "opening": "This nation was built by the hands and the hopes of those who came to these shores seeking a better life, and"},
    {"headline": "Billionaire Space Race Criticized Amid Poverty on Earth", "domain": "Economy",
     "opening": "The ingenuity of the few must not blind us to the needs of the many. While some race toward the stars,"},
    {"headline": "Public Schools Face Massive Teacher Shortage", "domain": "Social Welfare",
     "opening": "The schoolhouse is the workshop of democracy. When we cannot staff it, we are building our future on"},
    {"headline": "College Enrollment Drops for Fifth Year Running", "domain": "Social Welfare",
     "opening": "A nation that closes the doors of learning to its young people is a nation that has lost faith in"},
    {"headline": "Police Reform Bill Dies in Senate", "domain": "Government",
     "opening": "Justice delayed is justice denied, and justice denied is the seed of unrest. When the people cry out for reform and"},
    {"headline": "Insulin Costs Force Diabetics to Ration Medication", "domain": "Social Welfare",
     "opening": "That any citizen of this great republic should go without the medicine that sustains their life is"},
    {"headline": "Small Business Closures Hit Pandemic-Era Highs", "domain": "Economy",
     "opening": "The small business — the shop on the corner, the family enterprise — is the true engine of American prosperity, and"},
    {"headline": "Rent Control Debate Heats Up in Major Cities", "domain": "Social Welfare",
     "opening": "Shelter is not a commodity to be traded on an exchange. It is a necessity, and when the cost of a roof exceeds"},
    {"headline": "Military Recruitment Falls Short for Second Year", "domain": "Foreign Policy",
     "opening": "The defense of this nation has always rested upon the willingness of its citizens to serve, and when that willingness falters,"},
    {"headline": "Climate Refugees Expected to Reach 200 Million by 2050", "domain": "Conservation",
     "opening": "The displacement of millions by the changing climate is not a distant prophecy — it is happening now, and"},
    {"headline": "Big Tech Antitrust Case Goes to Trial", "domain": "Government",
     "opening": "When any industry grows so large that it controls the marketplace and silences competition, it is the duty of the people's government to"},
    {"headline": "National Debt Passes $35 Trillion", "domain": "Economy",
     "opening": "A great nation must be solvent — not merely in its treasury but in its obligations to the next generation, and"},
    {"headline": "Pandemic Preparedness Funding Cut by 40%", "domain": "Social Welfare",
     "opening": "We have learned at terrible cost that preparedness is not an expense — it is an investment. To cut the defenses against"},
    {"headline": "Trade War Escalates with New Tariffs on Both Sides", "domain": "Foreign Policy",
     "opening": "Trade among nations should be a bridge, not a battleground. When tariffs become weapons rather than tools,"},
    {"headline": "Youth Voter Registration Surges After Campus Organizing", "domain": "Government",
     "opening": "The young people of this nation have heard the call of their generation. When the youth march to the ballot box,"},
]

print(f'Prepared {len(MODERN_HEADLINES)} headlines for batch generation')

Prepared 50 headlines for batch generation


In [ ]:
# ── Batch generate all 50 responses ──
eval_responses = []

for i, h in enumerate(MODERN_HEADLINES):
    print(f'[{i+1}/{len(MODERN_HEADLINES)}] {h["headline"][:60]}...', end=' ', flush=True)

    # Policy-grounded response
    prompt_policy = f'[POLICY: {h["domain"]}] {h["opening"]}'
    policy_out = generator(
        prompt_policy,
        max_new_tokens=250,
        num_return_sequences=1,
        temperature=0.85,
        top_p=0.92,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2,
    )
    policy_text = re.sub(r'^\[POLICY: [^\]]+\]\s*', '', policy_out[0]['generated_text'])

    # Iconic quote
    prompt_iconic = f'[ICONIC] {h["opening"].split(".")[0]}.'
    iconic_out = generator(
        prompt_iconic,
        max_new_tokens=80,
        num_return_sequences=1,
        temperature=0.95,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.3,
    )
    iconic_text = re.sub(r'^\[ICONIC\]\s*', '', iconic_out[0]['generated_text'])
    # Trim iconic to first 2-3 sentences
    sentences = re.split(r'(?<=[.!?])\s+', iconic_text)
    iconic_text = ' '.join(sentences[:3])

    eval_responses.append({
        'id': f'headline_{i+1:02d}',
        'headline': h['headline'],
        'domain': h['domain'],
        'opening_prompt': h['opening'],
        'policy_response': policy_text,
        'iconic_quote': iconic_text,
    })
    print('done')

# Save to Drive for use in eval framework
EVAL_OUTPUT = '/content/drive/MyDrive/fdr_eval_responses.json'
with open(EVAL_OUTPUT, 'w') as f:
    json.dump(eval_responses, f, indent=2)

print(f'\nSaved {len(eval_responses)} response pairs to {EVAL_OUTPUT}')
print('Ready for Day 4 eval framework!')

[1/50] Fed Holds Interest Rates Steady Amid Inflation Fears... 

Both `max_new_tokens` (=250) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done
[2/50] Wall Street Hits Record High as Main Street Struggles... 

Both `max_new_tokens` (=250) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done
[3/50] Crypto Exchange Collapses, Billions in Customer Funds Missin... 

...

Both `max_new_tokens` (=250) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done
[36/50] Immigration Debate Paralyzes Congress... 

Both `max_new_tokens` (=250) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done
[37/50] Billionaire Space Race Criticized Amid Poverty on Earth... 

Both `max_new_tokens` (=250) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
# ── Preview a few responses ──
print('=' * 70)
print('  SAMPLE GENERATED RESPONSES')
print('=' * 70)

for r in eval_responses[:5]:
    print(f'\nHEADLINE: {r["headline"]}')
    print(f'DOMAIN:   {r["domain"]}')
    print(f'\nPOLICY RESPONSE:')
    print(f'  {r["policy_response"][:300]}...')
    print(f'\nICONIC QUOTE:')
    print(f'  "{r["iconic_quote"]}"')
    print('-' * 70)